# Medical Insurance Cost Analysis

**Analyzing the factors that drive medical insurance charges using regression and EDA techniques.**

## Project Overview

This project explores a medical insurance dataset to understand what factors most significantly influence insurance premiums. We examine demographic features like age, sex, BMI, number of children, smoking status, and region to uncover patterns in insurance charges.

The analysis follows a structured EDA pipeline: data validation, cleaning, univariate/bivariate analysis, statistical testing, and actionable insights.

## Learning Objectives

By completing this notebook, you will learn to:
1. Perform systematic exploratory data analysis on tabular data
2. Identify key drivers of a continuous target variable (charges)
3. Conduct statistical hypothesis tests (t-tests, ANOVA, correlation)
4. Create publication-quality visualizations
5. Interpret the interplay between categorical and numerical features
6. Recognize the outsized impact of lifestyle factors (smoking) on insurance costs

## Business / Research Problem

Insurance companies need to set premiums that accurately reflect risk. Understanding which factors contribute most to medical costs helps:
- **Insurers** price policies fairly and manage risk pools
- **Policyholders** understand what drives their premiums
- **Public health officials** identify high-cost demographic segments for intervention

**Key question:** *What are the primary factors that predict higher medical insurance charges, and how do they interact?*

## Why This Analysis Matters

Healthcare costs are among the largest expenses for individuals and employers. Understanding cost drivers enables:
- Better personal financial planning
- Evidence-based wellness program design
- Fairer actuarial pricing models
- Identification of modifiable risk factors (e.g., smoking, BMI) that can reduce costs

## Dataset Overview

| Feature | Type | Description |
|---------|------|-------------|
| `age` | Numeric | Age of the primary beneficiary |
| `sex` | Categorical | Gender (male/female) |
| `bmi` | Numeric | Body mass index |
| `children` | Numeric | Number of dependents covered |
| `smoker` | Categorical | Whether the person smokes (yes/no) |
| `region` | Categorical | US residential region (northeast, southeast, southwest, northwest) |
| `charges` | Numeric | Individual medical costs billed by insurance (target) |

**Size:** ~1,338 records

## Dataset Source & License Notes

- **Source:** [Kaggle - Medical Cost Personal Datasets](https://www.kaggle.com/datasets/mirichoi0218/insurance)
- **Original:** Machine Learning with R by Brett Lantz (inspired by)
- **License:** Open Database License (ODbL)
- **Usage:** Educational and research purposes

## Environment Setup

Install required packages if not already available.

In [ ]:
# Install dependencies
!pip install pandas numpy matplotlib seaborn scipy kagglehub -q

## Imports

Load all libraries needed for the analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print('All imports loaded successfully.')

## Configuration / Constants

In [ ]:
RANDOM_SEED = 42
KAGGLE_DATASET = 'mirichoi0218/insurance'
PALETTE = 'viridis'
SIGNIFICANCE_LEVEL = 0.05

np.random.seed(RANDOM_SEED)

## Dataset Download / Loading

We download the dataset directly from Kaggle using `kagglehub`.

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f'Dataset downloaded to: {path}')

# Find the CSV file
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
print(f'Loaded {len(df)} rows and {len(df.columns)} columns')
df.head()

## Data Validation Checks

Before any analysis, we verify data integrity: types, missing values, duplicates, and value ranges.

In [ ]:
print('=== Data Shape ===')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

print('\n=== Data Types ===')
print(df.dtypes)

print('\n=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Duplicate Rows ===')
print(f'Duplicates: {df.duplicated().sum()}')

print('\n=== Basic Statistics ===')
df.describe()

In [ ]:
# Check categorical columns
for col in ['sex', 'smoker', 'region']:
    print(f'{col}: {df[col].unique()}')
    print(f'  Value counts: {dict(df[col].value_counts())}')
    print()

## Data Cleaning

Handle duplicates, encode categorical variables for analysis, and verify no anomalous values exist.

In [ ]:
# Remove exact duplicates if any
initial_rows = len(df)
df = df.drop_duplicates()
print(f'Removed {initial_rows - len(df)} duplicate rows. Remaining: {len(df)}')

# Validate ranges
assert df['age'].between(0, 120).all(), 'Age out of range'
assert df['bmi'].between(5, 70).all(), 'BMI out of range'
assert df['children'].between(0, 20).all(), 'Children count out of range'
assert (df['charges'] > 0).all(), 'Charges must be positive'

print('All validation checks passed.')

In [ ]:
# Create useful derived features for analysis
df['bmi_category'] = pd.cut(df['bmi'],
                            bins=[0, 18.5, 25, 30, 100],
                            labels=['Underweight', 'Normal', 'Overweight', 'Obese'])

df['age_group'] = pd.cut(df['age'],
                         bins=[17, 30, 45, 60, 100],
                         labels=['18-30', '31-45', '46-60', '60+'])

print('Derived features created.')
df[['age', 'age_group', 'bmi', 'bmi_category']].head(10)

## Exploratory Data Analysis

Let's explore the overall distribution and characteristics of the data.

In [ ]:
print('=== Charges Summary ===')
print(f'Mean charges:   ${df["charges"].mean():,.2f}')
print(f'Median charges: ${df["charges"].median():,.2f}')
print(f'Std deviation:  ${df["charges"].std():,.2f}')
print(f'Min charges:    ${df["charges"].min():,.2f}')
print(f'Max charges:    ${df["charges"].max():,.2f}')
print(f'\nSkewness: {df["charges"].skew():.3f} (right-skewed)')
print(f'Kurtosis: {df["charges"].kurtosis():.3f}')

## Univariate Analysis

Examine the distribution of each feature individually.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Charges distribution
axes[0, 0].hist(df['charges'], bins=40, color='steelblue', edgecolor='white')
axes[0, 0].axvline(df['charges'].mean(), color='red', linestyle='--', label=f'Mean: ${df["charges"].mean():,.0f}')
axes[0, 0].axvline(df['charges'].median(), color='orange', linestyle='--', label=f'Median: ${df["charges"].median():,.0f}')
axes[0, 0].set_title('Distribution of Insurance Charges')
axes[0, 0].set_xlabel('Charges ($)')
axes[0, 0].legend()

# Age distribution
axes[0, 1].hist(df['age'], bins=30, color='coral', edgecolor='white')
axes[0, 1].set_title('Distribution of Age')
axes[0, 1].set_xlabel('Age')

# BMI distribution
axes[1, 0].hist(df['bmi'], bins=30, color='mediumseagreen', edgecolor='white')
axes[1, 0].axvline(25, color='orange', linestyle='--', label='Overweight threshold (25)')
axes[1, 0].axvline(30, color='red', linestyle='--', label='Obese threshold (30)')
axes[1, 0].set_title('Distribution of BMI')
axes[1, 0].set_xlabel('BMI')
axes[1, 0].legend()

# Children distribution
df['children'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 1], color='mediumpurple', edgecolor='white')
axes[1, 1].set_title('Distribution of Number of Children')
axes[1, 1].set_xlabel('Number of Children')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Categorical distributions
for i, col in enumerate(['sex', 'smoker', 'region']):
    df[col].value_counts().plot(kind='bar', ax=axes[i], color=sns.color_palette('Set2'), edgecolor='white')
    axes[i].set_title(f'Distribution of {col.title()}')
    axes[i].set_xlabel(col.title())
    axes[i].set_ylabel('Count')
    for p in axes[i].patches:
        axes[i].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                        ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

Explore how features relate to each other and to the target variable (charges).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Charges by smoker status
sns.boxplot(data=df, x='smoker', y='charges', ax=axes[0], palette='Set1')
axes[0].set_title('Charges by Smoking Status')

# Charges by sex
sns.boxplot(data=df, x='sex', y='charges', ax=axes[1], palette='Set2')
axes[1].set_title('Charges by Sex')

# Charges by region
sns.boxplot(data=df, x='region', y='charges', ax=axes[2], palette='Set3')
axes[2].set_title('Charges by Region')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: Age and BMI vs Charges, colored by smoker
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {'yes': 'red', 'no': 'steelblue'}
for status in ['no', 'yes']:
    subset = df[df['smoker'] == status]
    axes[0].scatter(subset['age'], subset['charges'], c=colors[status],
                   alpha=0.5, label=f'Smoker: {status}', s=20)
    axes[1].scatter(subset['bmi'], subset['charges'], c=colors[status],
                   alpha=0.5, label=f'Smoker: {status}', s=20)

axes[0].set_title('Age vs Charges (by Smoking Status)')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Charges ($)')
axes[0].legend()

axes[1].set_title('BMI vs Charges (by Smoking Status)')
axes[1].set_xlabel('BMI')
axes[1].set_ylabel('Charges ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Mean charges by age group and smoker status
pivot = df.groupby(['age_group', 'smoker'])['charges'].mean().unstack()
pivot.plot(kind='bar', figsize=(10, 6), color=['steelblue', 'coral'])
plt.title('Average Charges by Age Group and Smoking Status')
plt.xlabel('Age Group')
plt.ylabel('Average Charges ($)')
plt.legend(title='Smoker')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Feature-Specific Insights

Deep dive into the most influential features.

In [ ]:
# Smoker vs Non-Smoker comparison
smoker_stats = df.groupby('smoker')['charges'].agg(['mean', 'median', 'std', 'count'])
smoker_stats.columns = ['Mean Charges', 'Median Charges', 'Std Dev', 'Count']
smoker_stats['Mean Charges'] = smoker_stats['Mean Charges'].map('${:,.2f}'.format)
smoker_stats['Median Charges'] = smoker_stats['Median Charges'].map('${:,.2f}'.format)
smoker_stats['Std Dev'] = smoker_stats['Std Dev'].map('${:,.2f}'.format)
print('=== Smoker vs Non-Smoker ===')
print(smoker_stats)
print(f'\nSmokers pay approximately {df[df["smoker"]=="yes"]["charges"].mean() / df[df["smoker"]=="no"]["charges"].mean():.1f}x more than non-smokers.')

In [ ]:
# BMI category analysis
bmi_stats = df.groupby('bmi_category')['charges'].agg(['mean', 'median', 'count'])
bmi_stats.columns = ['Mean Charges', 'Median Charges', 'Count']
print('=== Charges by BMI Category ===')
print(bmi_stats.round(2))

fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(data=df, x='bmi_category', y='charges', hue='smoker', split=True, palette='Set1', ax=ax)
ax.set_title('Charges Distribution by BMI Category and Smoking Status')
ax.set_xlabel('BMI Category')
ax.set_ylabel('Charges ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Impact of number of children
children_stats = df.groupby('children')['charges'].agg(['mean', 'count'])
children_stats.columns = ['Mean Charges', 'Count']
print('=== Charges by Number of Children ===')
print(children_stats.round(2))

## Statistical Checks / Hypothesis Testing

We test whether observed differences in charges across groups are statistically significant.

In [ ]:
# Test 1: Do smokers pay significantly more than non-smokers?
smoker_charges = df[df['smoker'] == 'yes']['charges']
nonsmoker_charges = df[df['smoker'] == 'no']['charges']

t_stat, p_value = stats.ttest_ind(smoker_charges, nonsmoker_charges)
print('=== T-Test: Smoker vs Non-Smoker Charges ===')
print(f't-statistic: {t_stat:.4f}')
print(f'p-value: {p_value:.2e}')
print(f'Result: {"Significant" if p_value < SIGNIFICANCE_LEVEL else "Not significant"} difference at α={SIGNIFICANCE_LEVEL}')

In [ ]:
# Test 2: Do charges differ by sex?
male_charges = df[df['sex'] == 'male']['charges']
female_charges = df[df['sex'] == 'female']['charges']

t_stat, p_value = stats.ttest_ind(male_charges, female_charges)
print('=== T-Test: Male vs Female Charges ===')
print(f't-statistic: {t_stat:.4f}')
print(f'p-value: {p_value:.4f}')
print(f'Result: {"Significant" if p_value < SIGNIFICANCE_LEVEL else "Not significant"} difference at α={SIGNIFICANCE_LEVEL}')

In [ ]:
# Test 3: ANOVA - Do charges differ by region?
regions = [group['charges'].values for _, group in df.groupby('region')]
f_stat, p_value = stats.f_oneway(*regions)
print('=== One-Way ANOVA: Charges by Region ===')
print(f'F-statistic: {f_stat:.4f}')
print(f'p-value: {p_value:.4f}')
print(f'Result: {"Significant" if p_value < SIGNIFICANCE_LEVEL else "Not significant"} difference at α={SIGNIFICANCE_LEVEL}')

In [ ]:
# Correlation analysis for numerical features
numerical_cols = ['age', 'bmi', 'children', 'charges']
corr_matrix = df[numerical_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.3f',
            square=True, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

print('\nCorrelation with charges:')
for col in ['age', 'bmi', 'children']:
    r, p = stats.pearsonr(df[col], df['charges'])
    print(f'  {col}: r={r:.4f}, p={p:.2e}')

## Visual Analysis

Comprehensive visualizations to reveal patterns and interactions.

In [ ]:
# Pairplot colored by smoking status
g = sns.pairplot(df[['age', 'bmi', 'charges', 'smoker']], hue='smoker',
                 palette='Set1', diag_kind='kde', plot_kws={'alpha': 0.5, 's': 15})
g.figure.suptitle('Pairplot of Numerical Features by Smoking Status', y=1.02)
plt.show()

In [ ]:
# Charges distribution by age group, smoker, and BMI category
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=df, x='age_group', y='charges', hue='smoker', palette='Set1', ax=axes[0])
axes[0].set_title('Charges by Age Group and Smoking Status')

sns.boxplot(data=df, x='bmi_category', y='charges', hue='smoker', palette='Set1', ax=axes[1])
axes[1].set_title('Charges by BMI Category and Smoking Status')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: mean charges by age group and region
pivot_table = df.pivot_table(values='charges', index='age_group', columns='region', aggfunc='mean')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot_table, annot=True, fmt=',.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Average Charges by Age Group and Region')
plt.tight_layout()
plt.show()

## Key Findings

1. **Smoking is the dominant factor:** Smokers pay roughly 3-4x more in insurance charges than non-smokers
2. **Age increases charges linearly:** Older individuals have higher medical costs, with clear step-ups visible in scatter plots
3. **BMI interacts with smoking:** High BMI alone has moderate impact, but obese smokers face dramatically higher charges
4. **Region has minimal effect:** Regional differences in charges are small and not always statistically significant
5. **Sex differences are modest:** Males tend to have slightly higher charges, but the difference pales compared to smoking
6. **Number of children has weak correlation:** More children does not strongly predict higher individual charges
7. **Charges are right-skewed:** A small group of high-cost individuals (primarily smokers) pulls the mean well above the median

## Limitations

1. **Small dataset:** Only ~1,338 records — insufficient for robust subgroup analysis
2. **Limited features:** No information on pre-existing conditions, exercise habits, diet, or income
3. **US-only:** Results may not generalize to other countries with different healthcare systems
4. **Binary smoking variable:** Does not capture quantity smoked or years of smoking
5. **Cross-sectional data:** No temporal information to track cost changes over time
6. **Synthetic characteristics:** The data appears somewhat simplified and may not reflect real-world complexity

## Recommendations / Next Steps

1. **Build a predictive model:** Use linear regression, random forest, or gradient boosting to predict charges
2. **Feature engineering:** Create interaction terms (e.g., smoker × BMI) which clearly drive high costs
3. **Log-transform charges:** The right-skewed distribution suggests log-transformation may improve model performance
4. **Collect more data:** Additional features like exercise, diet, and medical history would improve analysis
5. **Segment analysis:** Build separate models or analyses for smokers vs. non-smokers

### How to Extend This Analysis
- Apply clustering (K-means) to identify natural risk groups
- Build an interactive dashboard to explore cost factors
- Compare with real insurance data if available
- Add machine learning for premium prediction

## Common Mistakes

1. **Ignoring skewness:** Treating charges as normally distributed without checking — always verify distribution shape before choosing statistical tests
2. **Ignoring interactions:** Looking at BMI or age in isolation misses the critical interaction with smoking status
3. **Using mean instead of median:** For skewed data, the median is a more representative central tendency measure
4. **Confusing correlation with causation:** High BMI correlates with higher charges, but the relationship may be confounded by other factors
5. **Not encoding categoricals properly:** Using label encoding for region implies an ordinal relationship that doesn't exist — use one-hot encoding instead
6. **Overlooking outliers:** The high-charge smokers aren't outliers to remove — they're a legitimate and important subpopulation

## Mini Challenge / Exercises

Test your understanding with these exercises:

1. **Exercise 1:** What is the average charge for a non-smoking female aged 30-45 in the southeast region?
2. **Exercise 2:** Create a new variable `high_cost` that is True if charges > $15,000. What percentage of smokers vs non-smokers are high-cost?
3. **Exercise 3:** Run a Mann-Whitney U test (non-parametric alternative to t-test) comparing smoker vs non-smoker charges. Do results differ from the t-test?
4. **Exercise 4:** Create a scatter plot of age vs charges with point size proportional to BMI. What patterns emerge?
5. **Exercise 5:** Build a simple linear regression predicting charges from age, bmi, and smoker status. What R² do you get?

In [ ]:
# Space for your exercise solutions

# Exercise 1:
# subset = df[(df['smoker'] == 'no') & (df['sex'] == 'female') & ...]

# Exercise 2:
# df['high_cost'] = df['charges'] > 15000

# Exercise 3:
# from scipy.stats import mannwhitneyu

# Exercise 4:
# plt.scatter(df['age'], df['charges'], s=df['bmi'])

# Exercise 5:
# from sklearn.linear_model import LinearRegression

## Final Summary / Key Takeaways

- **Smoking is the single most important predictor** of high insurance charges, dwarfing all other factors
- **Age and BMI both contribute** to higher charges, but their impact is amplified by smoking
- **The interaction between smoking and obesity** creates a "super high-cost" group that should be modeled separately
- **Geographic region has negligible impact** on individual charges in this dataset
- **For predictive modeling**, interaction features (smoker × BMI, smoker × age) would likely be the most valuable additions

This analysis demonstrates the importance of looking beyond simple univariate statistics and exploring how features interact to drive outcomes.